# BM25 Hybrid Retrieval Experiment

Tests whether adding BM25 to the retrieval stage improves over embedding-only similarity.
Uses Reciprocal Rank Fusion (RRF) to combine BM25 and embedding scores.

**Configurations tested:**
1. BM25 only
2. Embedding only (current baseline)
3. Hybrid BM25 + Embedding (RRF)
4. Hybrid → SVM → Classifier

All using the existing MiniLM-L6-v2 embeddings from HuggingFace Hub.

In [ ]:
# Option 1: If running locally alongside the repo
import sys; sys.path.insert(0, "../src")

# Option 2: In Colab, clone and add to path
# !git clone https://github.com/semajyllek/ctmatch.git
# import sys; sys.path.insert(0, "/content/ctmatch/src")

# Dependencies
!pip install -q sentence-transformers datasets scikit-learn transformers tqdm evaluate lxml rank-bm25


In [ ]:
import os
import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

DATA_ROOT = os.environ.get('CTMATCH_DATA_ROOT', '../data')

## Load data from HuggingFace Hub

In [ ]:
HF_ROOT = 'semaj83/ctmatch_ir'

def load_hf_text(path, is_text=False):
    ds = load_dataset(HF_ROOT, data_files=path)
    if is_text:
        return pd.DataFrame(ds['train'])
    arrays = [np.asarray(a['text'].split(','), dtype=float) for a in ds['train']]
    return pd.DataFrame(arrays)

print('Loading doc texts...')
doc_texts_df = load_hf_text('doc_texts.txt', is_text=True)
print(f'  {len(doc_texts_df)} documents')

print('Loading embeddings...')
doc_embeddings_df = load_hf_text('doc_embeddings.txt')
print(f'  shape: {doc_embeddings_df.shape}')

print('Loading index2docid...')
index2docid = load_hf_text('index2docid.txt', is_text=True)
print(f'  {len(index2docid)} mappings')

## Build BM25 index

In [ ]:
from ctmatch.matching.retrieval.bm25 import BM25Index

# Extract text strings from the dataframe
doc_text_list = [row[0] if isinstance(row, (list, tuple)) else str(row)
                 for row in doc_texts_df.values]

print(f'Building BM25 index over {len(doc_text_list)} documents...')
bm25_index = BM25Index(doc_text_list)
print('Done.')

## Load topics and relevance judgments

In [ ]:
from ctmatch.evaluation.eval_utils import (
    get_trec_topic2text, get_kz_topic2text,
    calc_first_positive_rank, calc_f1,
)

TREC_TOPIC_PATH = os.path.join(DATA_ROOT, 'trec_data/trec_21_topics.xml')
KZ_TOPIC_PATH = os.path.join(DATA_ROOT, 'kz_data/topics-2014_2015-description.topics')
TREC_REL_PATH = os.path.join(DATA_ROOT, 'trec_data/trec_21_judgments.txt')
KZ_REL_PATH = os.path.join(DATA_ROOT, 'kz_data/qrels-clinical_trials.txt')

# Load topics
topic2text = {}
if os.path.exists(TREC_TOPIC_PATH):
    topic2text.update(get_trec_topic2text(TREC_TOPIC_PATH))
if os.path.exists(KZ_TOPIC_PATH):
    topic2text.update(get_kz_topic2text(KZ_TOPIC_PATH))
print(f'Loaded {len(topic2text)} topics')

# Load relevance judgments
rel_dict = {}
for rel_path in [TREC_REL_PATH, KZ_REL_PATH]:
    if not os.path.exists(rel_path):
        continue
    with open(rel_path) as f:
        for line in f:
            tid, _, did, rel = line.split()
            rel_dict.setdefault(tid, {})[did] = int(rel)
print(f'Loaded judgments for {len(rel_dict)} topics')

## Helper: get doc indices from NCT IDs

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def get_doc_indices(doc_ids):
    indices = []
    for did in doc_ids:
        rows = np.where(index2docid['text'] == did)
        if len(rows[0]) > 0:
            indices.append(rows[0][0])
    return indices

def idx_to_nctid(idx):
    return index2docid.iloc[idx].values[0]

## Run retrieval experiments

In [ ]:
from ctmatch.matching.retrieval.embeddings import embedding_only_filter
from ctmatch.matching.retrieval.bm25 import bm25_filter
from ctmatch.matching.retrieval.hybrid import hybrid_filter
from numpy.linalg import norm

MAX_TOPICS = 50

def evaluate_retrieval(retrieval_fn, name, top_k=10):
    fprs, frrs, f1s = [], [], []
    topics_evaluated = 0
    
    for topic_id, topic_text in tqdm(list(topic2text.items())[:MAX_TOPICS], desc=name):
        if topic_id not in rel_dict:
            continue
        
        doc_ids = list(rel_dict[topic_id].keys())
        doc_set = get_doc_indices(doc_ids)
        if len(doc_set) == 0:
            continue
        
        ranked_indices = retrieval_fn(topic_text, doc_set)
        ranked_nctids = [idx_to_nctid(idx) for idx in ranked_indices]
        
        fpr, frr = calc_first_positive_rank(ranked_nctids, rel_dict[topic_id])
        f1 = calc_f1(ranked_nctids, rel_dict[topic_id])
        fprs.append(fpr)
        frrs.append(frr)
        f1s.append(f1)
        topics_evaluated += 1
    
    return {
        'name': name,
        'topics': topics_evaluated,
        'mean_fpr': np.mean(fprs),
        'mean_mrr': np.mean(frrs),
        'mean_f1': np.mean(f1s),
    }

In [ ]:
# 1. Embedding only
def emb_retrieve(topic_text, doc_set):
    topic_emb = embedding_model.encode([topic_text])[0]
    return embedding_only_filter(topic_emb, doc_embeddings_df, doc_set, top_n=len(doc_set))

emb_results = evaluate_retrieval(emb_retrieve, 'Embedding only')
print(emb_results)

In [ ]:
# 2. BM25 only
def bm25_retrieve(topic_text, doc_set):
    return bm25_filter(bm25_index, topic_text, doc_set, top_n=len(doc_set))

bm25_results = evaluate_retrieval(bm25_retrieve, 'BM25 only')
print(bm25_results)

In [ ]:
# 3. Hybrid BM25 + Embedding (RRF)
def hybrid_retrieve(topic_text, doc_set):
    topic_emb = embedding_model.encode([topic_text])[0]
    return hybrid_filter(
        bm25_index, topic_text,
        topic_emb, doc_embeddings_df,
        doc_set, top_n=len(doc_set),
    )

hybrid_results = evaluate_retrieval(hybrid_retrieve, 'Hybrid BM25+Emb (RRF)')
print(hybrid_results)

## Compare results

In [ ]:
comparison = pd.DataFrame([emb_results, bm25_results, hybrid_results])
comparison = comparison.set_index('name')[['mean_fpr', 'mean_mrr', 'mean_f1', 'topics']]
comparison.columns = ['Mean FPR ↓', 'Mean MRR ↑', 'Mean F1 ↑', 'Topics']
comparison.round(4)

## RRF sensitivity: tune k parameter

In [ ]:
k_results = []
for rrf_k in [10, 30, 60, 100, 200]:
    def hybrid_k(topic_text, doc_set, _k=rrf_k):
        topic_emb = embedding_model.encode([topic_text])[0]
        return hybrid_filter(
            bm25_index, topic_text,
            topic_emb, doc_embeddings_df,
            doc_set, top_n=len(doc_set), rrf_k=_k,
        )
    res = evaluate_retrieval(hybrid_k, f'RRF k={rrf_k}')
    k_results.append(res)

k_df = pd.DataFrame(k_results).set_index('name')[['mean_fpr', 'mean_mrr', 'mean_f1']]
k_df.columns = ['Mean FPR ↓', 'Mean MRR ↑', 'Mean F1 ↑']
k_df.round(4)